# big llm

In [36]:
from pathlib import Path
import pandas as pd

# =========================
# Пути
# =========================
BASE = Path(".")

sql_hr_big_llm_path = BASE / "sql-hr" / "retrieval_sql_hr_bigger_llm.csv"
judgments_path      = BASE / "llm_judge" / "judgments.csv"
gold_subset_path    = BASE / "llm_judge" / "gold_subset.csv"
queries_path        = BASE / "prepared" / "queries.csv"
pooled_path         = BASE / "prepared" / "pooled_candidates.csv"

output_missing_pairs_big_llm_path = BASE / "llm_judge" / "sql_hr_big_llm_missing_pairs.csv"

In [37]:
# =========================
# Загрузка
# =========================
sql_hr_results_big_llm = pd.read_csv(sql_hr_big_llm_path)
judgments = pd.read_csv(judgments_path)
gold_subset = pd.read_csv(gold_subset_path)
queries = pd.read_csv(queries_path)
pooled_candidates = pd.read_csv(pooled_path)

In [38]:
# =========================
# 1. Оставляем только gold queries
# =========================
gold_query_ids = gold_subset.loc[gold_subset["is_gold"] == True, "query_id"].unique()

sql_hr_results_big_llm_selected = sql_hr_results_big_llm[
    sql_hr_results_big_llm["query_id"].isin(gold_query_ids)
].copy()

print("sql_hr_results_selected shape:", sql_hr_results_big_llm_selected.shape)
print("unique gold queries in selected:", sql_hr_results_big_llm_selected["query_id"].nunique())


sql_hr_results_selected shape: (597, 8)
unique gold queries in selected: 40


In [39]:
# =========================
# 2. Нормализуем пары и проверяем дубли
# =========================
selected_pairs_sql_big_llm = (
    sql_hr_results_big_llm_selected[
        ["query_id", "candidate_id", "search_system", "rank", "score", "latency_ms", "retrieved_at"]
    ]
    .drop_duplicates(subset=["query_id", "candidate_id"])
    .copy()
)

judged_pairs = (
    judgments[["query_id", "candidate_id"]]
    .drop_duplicates()
    .copy()
)

print("selected_pairs:", selected_pairs_sql_big_llm.shape)
print("judged_pairs:", judged_pairs.shape)

selected_pairs: (597, 7)
judged_pairs: (1759, 2)


In [40]:
# =========================
# 3. Anti-join: находим пары без разметки
# =========================
missing_pairs_sql_big_llm = selected_pairs_sql_big_llm.merge(
    judged_pairs,
    on=["query_id", "candidate_id"],
    how="left",
    indicator=True
)

missing_pairs_sql_big_llm = missing_pairs_sql_big_llm[missing_pairs_sql_big_llm["_merge"] == "left_only"].drop(columns=["_merge"]).copy()

print("missing_pairs shape:", missing_pairs_sql_big_llm.shape)
print("missing coverage ratio:",
      round(len(missing_pairs_sql_big_llm) / len(selected_pairs_sql_big_llm), 4) if len(selected_pairs_sql_big_llm) else 0.0)

# Сохраняем просто пары + retrieval-мета
missing_pairs_sql_big_llm[['query_id', 'candidate_id', 'search_system']].to_csv(output_missing_pairs_big_llm_path, index=False)
print(f"saved: {output_missing_pairs_big_llm_path}")

missing_pairs shape: (463, 7)
missing coverage ratio: 0.7755
saved: llm_judge/sql_hr_big_llm_missing_pairs.csv


# agent rag

In [30]:
from pathlib import Path
import pandas as pd

# =========================
# Пути
# =========================
BASE = Path(".")

retrieval_rag_agent_path = BASE / "agent_rag" / "retrieval_rag_agent.csv"
judgments_path      = BASE / "llm_judge" / "judgments.csv"
gold_subset_path    = BASE / "llm_judge" / "gold_subset.csv"
queries_path        = BASE / "prepared" / "queries.csv"
pooled_path         = BASE / "prepared" / "pooled_candidates.csv"

output_missing_pairs_agent_rag_path = BASE / "llm_judge" / "agent_rag_missing_pairs.csv"

In [31]:
# =========================
# Загрузка
# =========================
retrieval_rag_agent = pd.read_csv(retrieval_rag_agent_path)
judgments = pd.read_csv(judgments_path)
gold_subset = pd.read_csv(gold_subset_path)
queries = pd.read_csv(queries_path)
pooled_candidates = pd.read_csv(pooled_path)

In [32]:
# =========================
# 1. Оставляем только gold queries
# =========================
gold_query_ids = gold_subset.loc[gold_subset["is_gold"] == True, "query_id"].unique()

retrieval_rag_agent_selected = retrieval_rag_agent[
    retrieval_rag_agent["query_id"].isin(gold_query_ids)
].copy()

print("retrieval_rag_agent_selected shape:", retrieval_rag_agent_selected.shape)
print("unique gold queries in selected:", retrieval_rag_agent_selected["query_id"].nunique())


retrieval_rag_agent_selected shape: (600, 7)
unique gold queries in selected: 40


In [33]:
# =========================
# 2. Нормализуем пары и проверяем дубли
# =========================
selected_pairs_rag_agent = (
    retrieval_rag_agent_selected[
        ["query_id", "candidate_id", "search_system", "rank", "score", "latency_ms", "retrieved_at"]
    ]
    .drop_duplicates(subset=["query_id", "candidate_id"])
    .copy()
)

judged_pairs = (
    judgments[["query_id", "candidate_id"]]
    .drop_duplicates()
    .copy()
)

print("selected_pairs:", selected_pairs_rag_agent.shape)
print("judged_pairs:", judged_pairs.shape)

selected_pairs: (593, 7)
judged_pairs: (1759, 2)


In [34]:
# =========================
# 3. Anti-join: находим пары без разметки
# =========================
missing_pairs_agent_rag = selected_pairs_rag_agent.merge(
    judged_pairs,
    on=["query_id", "candidate_id"],
    how="left",
    indicator=True
)

missing_pairs_agent_rag = missing_pairs_agent_rag[missing_pairs_agent_rag["_merge"] == "left_only"].drop(columns=["_merge"]).copy()

print("missing_pairs shape:", missing_pairs_agent_rag.shape)
print("missing coverage ratio:",
      round(len(missing_pairs_agent_rag) / len(selected_pairs_rag_agent), 4) if len(selected_pairs_rag_agent) else 0.0)

# Сохраняем просто пары + retrieval-мета
missing_pairs_agent_rag[['query_id', 'candidate_id', 'search_system']].to_csv(output_missing_pairs_agent_rag_path, index=False)
print(f"saved: {output_missing_pairs_agent_rag_path}")

missing_pairs shape: (254, 7)
missing coverage ratio: 0.4283
saved: llm_judge/agent_rag_missing_pairs.csv


In [35]:
missing_pairs_agent_rag

,query_id,candidate_id,search_system,rank,score,latency_ms,retrieved_at
0,Q_00001,C_01997,agent_rag,1.0,0.60,19056.299982,2026-04-05T16:23:55.597408
1,Q_00001,C_00550,agent_rag,2.0,0.50,19056.299982,2026-04-05T16:23:55.597408
2,Q_00001,C_01991,agent_rag,3.0,0.40,19056.299982,2026-04-05T16:23:55.597408
3,Q_00001,C_00815,agent_rag,4.0,0.30,19056.299982,2026-04-05T16:23:55.597408
4,Q_00001,C_01990,agent_rag,5.0,0.20,19056.299982,2026-04-05T16:23:55.597408
...,...,...,...,...,...,...,...
584,Q_00159,C_01940,agent_rag,7.0,0.89,28463.943528,2026-04-05T18:07:02.889138
585,Q_00159,C_13482,agent_rag,8.0,0.88,28463.943528,2026-04-05T18:07:02.889138
586,Q_00159,C_24252,agent_rag,9.0,0.87,28463.943528,2026-04-05T18:07:02.889138
591,Q_00159,C_21135,agent_rag,14.0,0.82,28463.943528,2026-04-05T18:07:02.889138


In [25]:
(retrieval_rag_agent[
    (retrieval_rag_agent['query_id'] == 'Q_00001') & (retrieval_rag_agent['candidate_id'] == 'C_01997')
    ])

,query_id,candidate_id,rank,score,latency_ms,retrieved_at,search_system
0,Q_00001,C_01997,1.0,0.6,19056.299982,2026-04-05T16:23:55.597408,agent_rag


In [24]:
(judgments[
    (judgments['query_id'] == 'Q_00001') & (judgments['candidate_id'] == 'C_01997')
    ])

,query_id,candidate_id,relevance_score,search_system,judge_model,judge_prompt,judge_comment


In [28]:
judgments.columns

Index(['query_id', 'candidate_id', 'relevance_score', 'search_system',
       'judge_model', 'judge_prompt', 'judge_comment'],
      dtype='object')

In [44]:
missing_pairs_sql_big_llm[['query_id', 'candidate_id']]

,query_id,candidate_id
5,Q_00001,C_31847
10,Q_00001,C_25963
12,Q_00001,C_36485
13,Q_00001,C_27299
14,Q_00001,C_25937
...,...,...
592,Q_00159,C_04819
593,Q_00159,C_14902
594,Q_00159,C_07595
595,Q_00159,C_39182


In [46]:
missing_pairs_agent_rag[['query_id', 'candidate_id']]

,query_id,candidate_id
0,Q_00001,C_01997
1,Q_00001,C_00550
2,Q_00001,C_01991
3,Q_00001,C_00815
4,Q_00001,C_01990
...,...,...
584,Q_00159,C_01940
585,Q_00159,C_13482
586,Q_00159,C_24252
591,Q_00159,C_21135


In [47]:
intersection = missing_pairs_sql_big_llm[['query_id', 'candidate_id']].merge(
    missing_pairs_agent_rag[['query_id', 'candidate_id']],
    on=['query_id', 'candidate_id'],
    how='inner'
)

len(intersection)

3

In [48]:
intersection

,query_id,candidate_id
0,Q_00022,C_39771
1,Q_00092,C_34567
2,Q_00142,C_39771
